# Steel Plate Surface Defect Classification
### A multiclass machine learning project using Random Forest

**Goal:** Predict the category of a steel plate surface fault from numerical inspection measurements.

**Dataset:** [Faulty Steel Plates (Kaggle)](https://www.kaggle.com/datasets/uciml/faulty-steel-plates) · 1,941 rows · 27 input features · 7 defect categories.

**Workflow:** Data review → target preparation → stratified split → Random Forest tuning → held-out evaluation → feature importance → save and reuse model.

> **Scope:** This dataset contains labeled steel plate defects. The model predicts *which defect category* is present; it does **not** predict vessel safety or PASS/FAIL status. The notebook is an educational portfolio project, not a production inspection system.

**Run instructions:** Attach the *Faulty Steel Plates* dataset in Kaggle and run all cells from top to bottom. The file is expected to be named `faults.csv`. The notebook also works in Colab when `faults.csv` is uploaded to the working directory.

## 1. Setup and imports

Load the libraries used for data analysis, model training, evaluation, visualization, and model persistence.

In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)


## 2. Load and inspect the dataset

The loader checks standard Kaggle and Google Colab locations. In Kaggle, attach the dataset to the notebook before running.

In [2]:
csv_candidates = [Path("faults.csv"), Path("/content/faults.csv")]
kaggle_input = Path("/kaggle/input")

if kaggle_input.exists():
    csv_candidates.extend(kaggle_input.rglob("faults.csv"))

csv_path = next((path for path in csv_candidates if path.is_file()), None)
if csv_path is None:
    raise FileNotFoundError(
        "faults.csv was not found. Attach the Faulty Steel Plates dataset "
        "in Kaggle, or upload faults.csv to your Colab working directory."
    )

load = pd.read_csv(csv_path)
print(f"Dataset: {csv_path}")
print(f"Shape: {load.shape}")
display(load.head())


Dataset: /kaggle/input/datasets/harnilkadia/faults/faults.csv
Shape: (1941, 34)


,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
0,42,50,270900,270944,267,17,44,24220,76,108,...,0.8182,-0.2913,0.5822,1,0,0,0,0,0,0
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.7931,-0.1756,0.2984,1,0,0,0,0,0,0
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.6667,-0.1228,0.2150,1,0,0,0,0,0,0
3,853,860,369370,369415,176,13,45,18996,99,126,...,0.8444,-0.1568,0.5212,1,0,0,0,0,0,0
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.9338,-0.1992,1.0000,1,0,0,0,0,0,0


In [3]:
print("Missing values:", int(load.isna().sum().sum()))
print("Duplicate rows:", int(load.duplicated().sum()))
print("Number of columns:", len(load.columns))


Missing values: 0
Duplicate rows: 0
Number of columns: 34


## 3. Understand the seven defect labels

The original target uses seven binary indicator columns. Before converting them, verify that each steel plate has **exactly one** active defect label.

In [ ]:
defect_cols = [
    "Pastry",
    "Z_Scratch",
    "K_Scatch",
    "Stains",
    "Dirtiness",
    "Bumps",
    "Other_Faults",
]

print("Examples in each category:")
display(load[defect_cols].sum().sort_values(ascending=False).rename("Count"))

label_count_per_row = load[defect_cols].sum(axis=1)
print("Number of active labels per row:")
display(label_count_per_row.value_counts().sort_index().rename("Rows"))

if not label_count_per_row.eq(1).all():
    raise ValueError("Expected exactly one defect label per row.")


## 4. Create one categorical target and separate features

Convert the seven indicators into one `Target` containing the defect name. The nested loops below preserve the original learning approach. Exclude the seven indicator columns from the model inputs to **prevent target leakage**.

In [ ]:
load["Target"] = None

for col in defect_cols:
    for row_index, value in load[col].items():
        if value == 1:
            load.loc[row_index, "Target"] = col

X = load.drop(columns=defect_cols + ["Target"])
y = load["Target"]

print("X shape:", X.shape)
print("y shape:", y.shape)
display(y.value_counts().rename("Samples"))


## 5. Stratified train/test split

Reserve 20% of the rows as a held-out test set. Stratification preserves approximately the same class proportions in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training labels:", y_train.shape)
print("Test labels:", y_test.shape)

split_counts = pd.DataFrame({
    "Train": y_train.value_counts(),
    "Test": y_test.value_counts(),
}).fillna(0).astype(int)
display(split_counts)


## 6. Tune the Random Forest with cross-validation

Use **RandomizedSearchCV** to sample 15 hyperparameter combinations, evaluated with stratified 5-fold cross-validation. **Macro F1** gives each fault category equal weight, which matters because the dataset is imbalanced.

The fixed random seed makes the search reproducible. Scores may differ from an earlier run where the search seed was unspecified.

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=1)

param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 10, 20],
    "max_features": ["sqrt", 0.5],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=15,
    cv=cv,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)
print("Best cross-validation Macro F1:", round(grid.best_score_, 4))
print("Best hyperparameters:", grid.best_params_)


## 7. Evaluate the selected model on the held-out test set

With `refit=True` (the default), the search automatically fits its best configuration on the **full training split**. We evaluate this fitted model on `X_test` without retraining it on test samples.

**Note:** The test score is an estimate of performance on unseen examples, not a guarantee of production performance.

In [ ]:
new_model = grid.best_estimator_
pred = new_model.predict(X_test)

accuracy = accuracy_score(y_test, pred)
macro_f1 = f1_score(y_test, pred, average="macro")

print(f"Test accuracy: {accuracy:.4f} ({accuracy:.2%})")
print(f"Test Macro F1: {macro_f1:.4f}")
print("Classification report:")
print(classification_report(y_test, pred, zero_division=0))


In [ ]:
class_labels = sorted(y.unique())
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred,
    labels=class_labels,
    xticks_rotation=45,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title("Random Forest — Test Set Confusion Matrix")
plt.tight_layout()
plt.show()


### Interpreting the results

Read **recall** for categories where missed detections matter, and **precision** for categories where false alarms matter. Pay particular attention to `Pastry` and other less reliably classified faults. Here, all target categories denote faults: mistaking one fault type for another is **not** the same as predicting a defect-free plate.

## 8. Built-in Random Forest feature importance

Tree-based importance measures how much features reduce impurity within the fitted forest. It is useful for exploration but does not establish a causal relationship or the direction of an effect.

In [ ]:
rf_importance = (
    pd.DataFrame({
        "Feature": X.columns,
        "Importance": new_model.feature_importances_,
    })
    .sort_values("Importance", ascending=False)
)

display(rf_importance.head(10))

top_rf = rf_importance.head(10).sort_values("Importance")
plt.figure(figsize=(10, 6))
plt.barh(top_rf["Feature"], top_rf["Importance"])
plt.xlabel("Impurity-based importance")
plt.ylabel("Feature")
plt.title("Top 10 Features — Random Forest")
plt.tight_layout()
plt.show()


## 9. Permutation importance on separate validation data

For this analysis only, split the **training set** again and fit a **fresh copy** of the tuned estimator on the smaller subset. Evaluate how much validation Macro F1 decreases when a feature is shuffled.

Using `clone(new_model)` ensures this experiment does **not** overwrite the final model trained on the full `X_train`. Feature rankings can be unstable when predictors are correlated or the validation set is small.

In [ ]:
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train,
)

validation_model = clone(new_model)
validation_model.fit(X_subtrain, y_subtrain)

perm_result = permutation_importance(
    validation_model,
    X_val,
    y_val,
    scoring="f1_macro",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

perm_df = (
    pd.DataFrame({
        "Feature": X.columns,
        "Mean_Importance": perm_result.importances_mean,
        "Std_Importance": perm_result.importances_std,
    })
    .sort_values("Mean_Importance", ascending=False)
)

display(perm_df.head(10))


In [ ]:
top_perm = perm_df.head(10).sort_values("Mean_Importance")
plt.figure(figsize=(10, 6))
plt.barh(
    top_perm["Feature"],
    top_perm["Mean_Importance"],
    xerr=top_perm["Std_Importance"],
)
plt.xlabel("Mean decrease in validation Macro F1")
plt.ylabel("Feature")
plt.title("Top 10 Features — Permutation Importance")
plt.tight_layout()
plt.show()


## 10. Save, load, and use the trained model

Save the selected estimator fitted on the full `X_train` and verify that it can be loaded and used to predict defect categories. A model receiving new data must receive **the same feature names and order** as the training data.

**Security:** Only load `.pkl` files that you created yourself or received from a trusted source.

In [ ]:
model_path = "steel_defect_model.pkl"
joblib.dump(new_model, model_path)
loaded_model = joblib.load(model_path)

sample_features = X_test.head(5)
sample_predictions = loaded_model.predict(sample_features)

sample_results = pd.DataFrame({
    "Actual": y_test.loc[sample_features.index].to_numpy(),
    "Predicted": sample_predictions,
}, index=sample_features.index)
sample_results.index.name = "Original_Row_Index"

display(sample_results)
print(f"Saved model to: {model_path}")


## 11. Conclusions and limitations

**What this notebook demonstrates:**

- Transforming seven mutually exclusive binary indicators into a single multiclass target.
- Preventing target leakage when creating 27 input features.
- Building a stratified hold-out set and tuning a Random Forest with macro F1.
- Reading per-class precision, recall, F1, and a confusion matrix.
- Comparing impurity-based and permutation feature importance.
- Saving and reloading a trained model for reuse.

**Limitations:** The data represents known steel plate defect classes, not vessel quality acceptance or zero-defect detection. Small classes have relatively few test examples. Performance should be checked on additional unseen manufacturing data before any operational deployment.

**Potential extensions:** Compare another model family, explore class imbalance, and package the predictor behind a simple interface. These are optional next steps rather than requirements for this portfolio notebook.

**Dataset attribution:** [Faulty Steel Plates — Kaggle](https://www.kaggle.com/datasets/uciml/faulty-steel-plates); [Steel Plates Faults — UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/198/steel+plates+faults).